# Reproducing Seyedolmohadesin et al. (2026) from dandiset 000981

Paper. Seyedolmohadesin M, Fu X, Torkashvand M, Rasouli S, Lang S, Li L, Kalinski C, Cook S,
Schroeder F, Yemini E, Venkatachalam V (2026). Whole-brain chemosensory responses of both
*C. elegans* sexes. *Current Biology* 36, 4803-4816.
[doi:10.1016/j.cub.2026.08.040](https://doi.org/10.1016/j.cub.2026.08.040). The claims file
(`claims/seyedolmohadesin-2026.yaml`) was written from the version of record, main text, STAR
Methods and the supplement's Figures S1 to S20 with their legends.

Dandiset. [DANDI:000981](https://dandiarchive.org/dandiset/000981/0.260910.1747), version
0.260910.1747 (doi:10.48324/dandi.000981/0.260910.1747). The paper cites the dandiset without a
version. This is the only published version ("First public release after publication",
2026-09-10); the draft lists the same 47 assets and the same total size, so the most recent
published version is pinned.

What the dandiset holds. One NWB file per animal, 47 in all: the raw GCaMP and NeuroPAL
volumes, the NeuroPAL identities, a table of calcium traces of the identified head neurons
(`ActivityTraces`) and the stimulus annotations (`StimulusInfo`). Only the traces, labels,
stimulus annotations and subject metadata are read here; the image volumes are not.

Paper's code. Method details the claims file does not give are taken from the paper, and
where the Methods leave them open from the code the claims file cites
(github.com/venkatachalamlab/whole-brain-chemosensory-responses at commit b0dbdbf); the helper
`chemo.py` says where each parameter comes from.

Outside inputs. The connectome claims ("with inputs") use the chemical connectomes of White
et al. (1986) as annotated by Cook et al. (2019) and of the male (Cook et al. 2019), from that
paper's supplement (doi:10.1038/s41586-019-1352-7), and datasets 7 and 8 of Witvliet et al. (2021)
from its supplement (doi:10.1038/s41586-021-03778-8), read into memory from the journal's servers.
The paper's own merged table is not public.

Random numbers are drawn only in the timing check of the data check, from a generator with a
fixed seed.

| Claim | Data | Status |
|---|---|---|
| recordings-animals | dandiset | tested |
| fig5b-responsive-classes | dandiset | tested |
| fig5b-dimorphic-classes | dandiset | tested |
| fig5b-layer-counts | dandiset | tested (layer assignment assumed) |
| fig3-each-stimulus-mixed | dandiset | tested |
| sensory-dimorphic-and-monomorphic | dandiset | tested |
| layers-sensory-most-dimorphic | dandiset | tested |
| fig3a-osmotic-onset-examples | dandiset | tested |
| fig2b-bag-high-odor-monomorphic | dandiset | tested |
| fig3a-ash-stimulus-dependent | dandiset | tested |
| fig3a-aser-salt-concentration | dandiset | tested |
| fig2e-i2-dimorphic | dandiset | tested |
| pharynx-dimorphic-food-salt-pheromone | dandiset | tested |
| fig2-awa-awb-nociceptive | dandiset | tested |
| fig2-adf-m2-switch | dandiset | tested |
| male-specific-unresponsive | dandiset | tested |
| tail-unresponsive | dandiset | blocked: no traces of tail neurons in the dandiset |
| figs2f-left-right-similar | dandiset | tested |
| figs3e-pattern-robust | dandiset | tested |
| figs2e-no-order-effect | dandiset | tested |
| figs3fg-state-dependence-interneurons | dandiset | tested |
| figs2c-aib-state | dandiset | tested |
| figs8-state-insensitive | dandiset | tested |
| fig4c-functional-connectivity-similar | dandiset | tested |
| fig4-stimuli-correlate-sensory | dandiset | tested |
| fig4-pharynx-uncorrelated | dandiset | tested |
| fig4-functional-dimorphism-small | dandiset | tested |
| connectome-link | with inputs | tested |
| connectome-normalization-sem | with inputs | tested |
| fig4d-functional-synaptic-uncorrelated | with inputs | tested |
| fig4f-synaptic-top-not-functional | with inputs | tested |
| fig4g-functional-top-not-synaptic | with inputs | tested |
| figs20-multihop-no-match | with inputs | tested |
| figs1-behavior | dandiset | blocked: the drop-test data are not in the dandiset |

In [ ]:
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy import optimize, stats
from statsmodels.stats.multitest import multipletests

import chemo

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
raw = chemo.load_all()
print(f"loaded {len(raw)} files from dandiset {chemo.DANDISET_ID} version {chemo.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "subject": r["subject"], "sex_field": r["sex_field"], "description": r["description"],
    "strain": r["strain"], "sex": r["sex"], "traces": r["activity"].shape[0],
    "frames": r["activity"].shape[1], "volumes": r["n_volumes"], "rate": r["rate"],
    "labels_unique": len(set(r["neuron"])), "neuropal_ids": len([x for x in r["neuropal_ids"] if x]),
    "stim_unit": r["stim_unit"], "n_cue_annotations": len(r["cue_frames"]),
    "n_ctx_annotations": r["stim_names"].count("CTX")} for r in raw])
print("Recordings by sex as given in the subject description:")
print(info.sex.value_counts().to_string())
print("\nSubject sex field values:", dict(Counter(info.sex_field)), "| strain values:", dict(Counter(info.strain)))
print("Sex letter in the subject id agrees with the description:",
      all((s.split("-")[1][0] == "h") == (x == "hermaphrodite") for s, x in zip(info.subject, info.sex)))
print("\nFrames per trace:", dict(Counter(info.frames)), "| image volumes:", dict(Counter(info.volumes)),
      "| stated rate (volumes/s):", dict(Counter(info.rate)))
print("Duration at 2.66 volumes/s: %.1f min (Methods: 31 min at 2.67 volumes/s)" % (4979 / 2.66 / 60))
print("\nTraces per file: min %d, median %d, max %d" % (info.traces.min(), info.traces.median(), info.traces.max()))
print("Files whose labels are not unique:", info[info.traces != info.labels_unique].subject.tolist())

The archive's subject `sex` field is "O" (other) for every file, and the strain is empty; the sex
is stated only in the subject description and the subject id ("h" or "m"), which agree: 22
hermaphrodites and 25 males, as the paper says. The traces have 4979 frames, two fewer than the
image series; 4979 frames at 2.66 volumes/s are 31.2 minutes, as the Methods say. Two files hold
382 traces with every label twice; they are examined below.

In [ ]:
by_subject = {r["subject"]: r for r in raw}
h3, m3 = by_subject["20220403-h3"], by_subject["20220403-m3"]
print("20220403-h3 and 20220403-m3: activity tables identical:",
      np.array_equal(h3["activity"], m3["activity"], equal_nan=True),
      "| labels identical:", h3["neuron"] == m3["neuron"],
      "| stimulus orders identical:", h3["order"] == m3["order"])
print("NeuroPAL identities: h3 %d, m3 %d; male-specific identities in m3's NeuroPAL segmentation: %s"
      % (len([x for x in h3["neuropal_ids"] if x]), len([x for x in m3["neuropal_ids"] if x]),
         sorted(x for x in m3["neuropal_ids"] if x[:3] in ("CEM", "MCM"))))
half = h3["neuron"].index(h3["neuron"][0], 1)   # the first label recurs where the second block starts
first, second_block = h3["neuron"][:half], h3["neuron"][half:]
print(f"Blocks: rows 0-{half - 1} ({len(first)} traces, {len(set(first))} labels) and rows {half}-{len(h3['neuron']) - 1} "
      f"({len(second_block)} traces, {len(set(second_block))} labels); labels only in the second block: {sorted(set(second_block) - set(first))}; "
      f"only in the first: {sorted(set(first) - set(second_block))}")
second = {n: i for i, n in enumerate(second_block, start=half)}
same_lab_r = [np.corrcoef(np.nan_to_num(h3["activity"][i, 100:]), np.nan_to_num(h3["activity"][second[n], 100:]))[0, 1]
              for i, n in enumerate(first) if n in second]
print("Median r between the two traces of a label: %.2f (range %.2f to %.2f)" % (np.median(same_lab_r), np.min(same_lab_r), np.max(same_lab_r)))

# Which stimulus order does each half follow? Compare each half's mean response per stimulus with the
# median response of the same neuron in the other 45 animals, using h3's or m3's stimulus order.
def stim_pattern(rec, rows=None):
    d = chemo.responses(rec, rows)
    return d.groupby(["neuron", "stimulus"]).avg_during.mean().clip(-2, 5).unstack()

others = [r for r in raw if r["subject"] not in ("20220403-h3", "20220403-m3")]
reference = pd.concat([stim_pattern(r) for r in others], keys=range(len(others))).groupby(level=1).median()

def match_to_reference(pattern):
    common = pattern.index.intersection(reference.index)
    return np.mean([np.corrcoef(pattern.loc[common, s], reference.loc[common, s])[0, 1] for s in chemo.STIMULI])

for name, rows in [("first block", slice(0, half)), ("second block", slice(half, None))]:
    for oname, order in [("h3 order", h3["order"]), ("m3 order", m3["order"])]:
        rec = dict(h3, order=order)
        print(f"{name}, {oname}: mean r with the other animals' responses = {match_to_reference(stim_pattern(rec, rows)):.3f}")

Files 20220403-h3 (hermaphrodite) and 20220403-m3 (male) hold the same trace table of 382 rows in
two blocks. The second block has the male-specific CEM and MCM neurons; the first has none. Read
with h3's stimulus order, the first block responds to the stimuli as the other animals do, and
much less so with m3's order; the second block matches the other animals with m3's order and not
with h3's. So the table holds h3's traces (first block) and m3's (second block), and each file
carries both. Below, h3 is analysed on the first block and m3 on the second; that the blocks are
these animals' is what the responses and the male-specific labels show, though the files do not
say so.

In [ ]:
recs = []
for r in raw:
    rows = None
    if r["subject"] == "20220403-h3":
        rows = slice(0, half)
    if r["subject"] == "20220403-m3":
        rows = slice(half, None)
    recs.append((r, rows))
print("Recordings used:", Counter(r["sex"] for r, _ in recs))

frames = pd.DataFrame([r["cue_frames"] - chemo.ON_STIM for r, _ in recs])
print("\nStimulus annotation frame minus the code's onset frame (config.py ON_stim), over all recordings:",
      dict(Counter(frames.values.ravel())))
print("Largest annotated frame: %d (the unit is given as '%s')" % (max(r["stim_frames"].max() for r, _ in recs),
                                                                  recs[0][0]["stim_unit"]))
ok_orders = all(len(r["cue_frames"]) == 30 and [n for n in r["stim_names"] if n != "CTX"][:10] ==
                [n for n in r["stim_names"] if n != "CTX"][10:20] == [n for n in r["stim_names"] if n != "CTX"][20:30]
                and sorted(r["order"]) == sorted(chemo.STIMULI) for r, _ in recs)
print("Every recording: 30 cue presentations, 10 stimuli in one order repeated three times:", ok_orders)
print("Distinct stimulus orders:", len(set(tuple(r["order"]) for r, _ in recs)), "of", len(recs))
ctx_gap = Counter()
for r, _ in recs:
    names, fr = r["stim_names"], r["stim_frames"]
    for i, n in enumerate(names[:-1]):
        if n != "CTX":
            ctx_gap[int(fr[i + 1] - fr[i])] += 1
print("Frames from a cue annotation to the next (CTX) annotation:", dict(ctx_gap))

Every recording has 30 cue presentations, its own order of the 10 stimuli repeated three times,
as the Methods say. The annotation times are labelled seconds but are frame numbers (they run to
4980, and 4980 s would be 83 minutes). Each cue annotation is followed 40 frames (15 s) later by a
CTX (buffer) annotation, the stimulus offset. The cue annotations lie 1 to 3 frames after the onset
frames the paper's code uses (config.py); the analysis uses the code's frames, as the claims file
(RESPONSE) says.

In [ ]:
act = np.concatenate([r["activity"][rows if rows else slice(None)].ravel() for r, rows in recs])
print("Trace values: min %.2f, max %.2f, median %.2f; zeros %.3f; missing %.4f"
      % (np.nanmin(act), np.nanmax(act), np.nanmedian(act), np.mean(act == 0), np.mean(np.isnan(act))))
nan_first = sum(np.isnan(r["activity"][rows if rows else slice(None)][:, :96]).all() for r, rows in recs)
print("Recordings whose first 96 frames are missing in every trace:", nan_first)
blank = []
for r, rows in recs:
    A = r["activity"][rows if rows else slice(None)]
    allrow = np.isnan(A).mean(0) > 0.9
    allrow[:96] = False
    if allrow.any():
        blank.append((r["subject"], int(allrow.sum())))
print("Recordings with other frames missing in (nearly) every trace (frames):", blank)

The traces are not documented beyond "list of reading at each frame". Their values (0 to 255,
often two decimals) are consistent with the mean of the 15 brightest pixels the Methods describe,
on a background-subtracted 8-bit scale. Zeros are single frames of dim neurons, not gaps. Missing
values appear as whole blocks: the first 96 frames in some recordings (the code blanks these
frames before computing correlations) and time ranges in others, consistent with the code's
per-recording ranges to ignore (its spreadsheet is not public).

In [ ]:
# Timing check: does each recording's stimulus order line up with its responses? For each animal,
# the mean onset response of each neuron to each stimulus is correlated with the median over the
# other animals; the same is done with the stimulus labels permuted within the cycle (200
# permutations, seeded).
rng = np.random.default_rng(20261110)
pat = {r["subject"]: stim_pattern(r, rows) for r, rows in recs}
rows_out = []
for subj, p in pat.items():
    ref = pd.concat([q for s, q in pat.items() if s != subj], keys=range(len(pat) - 1)).groupby(level=1).median()
    common = p.index.intersection(ref.index)
    P, Q = p.loc[common, chemo.STIMULI].values, ref.loc[common, chemo.STIMULI].values
    fin = np.isfinite(P).all(1) & np.isfinite(Q).all(1)
    P, Q = P[fin], Q[fin]
    true = np.nanmean([np.corrcoef(P[:, i], Q[:, i])[0, 1] for i in range(10)])
    perm = [np.nanmean([np.corrcoef(P[:, j], Q[:, i])[0, 1] for i, j in enumerate(rng.permutation(10)) if i != j] or [np.nan])
            for _ in range(200)]
    rows_out.append({"subject": subj, "r_true_labels": true, "r_permuted_mean": np.nanmean(perm),
                     "frac_perm_ge_true": np.mean(np.array(perm) >= true)})
timing = pd.DataFrame(rows_out)
print(timing.round(3).to_string(index=False))
print("\nRecordings where the true labels beat every permutation:", int((timing.frac_perm_ge_true == 0).sum()), "of", len(timing))

In 44 of 47 recordings the responses read with the recording's own stimulus order match the other
animals' better than under every one of 200 permuted orders. In 20220630-h6 and 20221014-m3, 0.5%
and 2% of permutations do as well; in 20221014-m2, 17.5% do (r 0.63 against a permutation mean of
0.60), so for that recording the shared clock of annotations and traces is consistent with the
data but not established by this check. This contrast between stimuli is what shows that the
annotations and the traces share a clock to within the response windows.

In [ ]:
df = pd.concat([chemo.responses(r, rows) for r, rows in recs], ignore_index=True)
print("Neuron-presentation responses:", len(df))
subs = sorted(set(df.subclass))
shared = [s for s in subs if s not in chemo.MALE_SPECIFIC]
canon = sorted(set(map(chemo.canonical, shared)))
print("Subclasses: %d sex-shared (paper: 110) and male-specific %s" % (len(shared), sorted(chemo.MALE_SPECIFIC & set(subs))))
print("Canonical classes: %d (paper: 89)" % len(canon))
print("Layer assignment (assumed; totals in the paper 24, 36, 15, 14):")
for layer, cs in chemo.LAYER_CLASSES.items():
    print(f"  {layer} ({len(cs)}): {', '.join(cs)}")
assert set(canon) == set(chemo.LAYER)
per_animal_neurons = df.groupby(["subject", "sex"]).neuron.nunique()
print("\nIdentified neurons per animal, by sex:")
print(per_animal_neurons.groupby(level=1).describe().round(1).to_string())
ms = df[df.subclass.isin(chemo.MALE_SPECIFIC)].groupby("sex").subject.nunique()
print("\nAnimals with male-specific neuron traces, by sex:", ms.to_dict())
tail = sorted({x for r, _ in recs for x in r["neuropal_ids"] if x[:2] in ("PH", "PV", "PL", "DV", "LU", "PD", "PQ") or x[:3] == "ALN"})
print("Tail neuron identities in the NeuroPAL segmentations (examples):", tail[:15])
print("Tail neurons among the traces:", sorted(set(df.neuron) & set(tail)))
print("Responses by sex, cycle (counts):")
print(pd.crosstab(df.sex, df.cycle).to_string())
for c in ["avg_before", "avg_during", "avg_after", "avg_off"]:
    print(f"{c}: 1st, 50th, 99th percentile = {np.nanpercentile(df[c], [1, 50, 99]).round(3)}, max {np.nanmax(df[c]):.1f}")

The labels give 110 sex-shared subclasses, as the paper's Figure 4 uses, and 89 canonical classes
after the code's collapsing, as Figure 5 counts. The male-specific CEM and MCM neurons have traces
in males only. The NeuroPAL segmentations identify tail neurons (PHA, PLM, PVQ and others), but
no tail neuron has a trace. A few responses are very large where F0 is near zero; the outlier rule
below is the paper's remedy.

## Exclusion and the class tests used by most claims

The claims file names two readings of the exclusion (EXCLUSION): (a) the Methods' 5 IQR rule on the
three window means; (b) also the code's filters, as far as the dandiset allows (more than 7 missing
frames of 120, and the same rule on the window ranges; the code's identification confidence is not
in the dandiset). Each claim below is run under both.

In [ ]:
X = {}
for rd in ("a", "b"):
    e = chemo.exclude(df, rd)
    X[rd] = {"e": e, "rt": chemo.responsive_table(e),
             "dt1": chemo.dimorphism_table(e, 1), "dt2": chemo.dimorphism_table(e, 2),
             "dt3": chemo.dimorphism_table(e, 3)}
    dropped = 1 - len(e) / len(df)
    print(f"Exclusion reading ({rd}): {len(df) - len(e)} of {len(df)} responses dropped ({dropped:.1%}); "
          f"by sex: {(1 - e.sex.value_counts() / df.sex.value_counts()).round(3).to_dict()}")


def class_calls(rd, thr=0.05, strict=False, rep="dt1"):
    """Per sex-shared subclass and canonical class: responsive, dimorphic, and per-cue calls."""
    rt, dt = X[rd]["rt"], X[rd][rep]
    sig = (lambda p: p < thr) if strict else (lambda p: p <= thr)
    rt = rt.assign(sig=sig(rt.p)); dt = dt.assign(sig=sig(dt.p))
    rt = rt[~rt.subclass.isin(chemo.MALE_SPECIFIC)]; dt = dt[~dt.subclass.isin(chemo.MALE_SPECIFIC)]
    resp_cue = rt.groupby(["subclass", "cue"]).sig.any().unstack().reindex(columns=chemo.CUES).fillna(False)
    dim_cue = dt.groupby(["subclass", "cue"]).sig.any().unstack().reindex(columns=chemo.CUES).fillna(False)
    sub = pd.DataFrame({"responsive": resp_cue.any(axis=1), "dimorphic": dim_cue.any(axis=1)}).fillna(False)
    sub["canonical"] = sub.index.map(chemo.canonical)
    can_resp_cue = resp_cue.groupby(chemo.canonical).any()
    can_dim_cue = dim_cue.groupby(chemo.canonical).any()
    can = sub.groupby("canonical")[["responsive", "dimorphic"]].any()
    can["both"] = can.responsive & can.dimorphic
    can["layer"] = can.index.map(chemo.LAYER)
    return {"sub": sub, "can": can, "resp_cue": resp_cue, "dim_cue": dim_cue,
            "can_resp_cue": can_resp_cue, "can_dim_cue": can_dim_cue}


C = {(rd, strict): class_calls(rd, strict=strict) for rd in "ab" for strict in (False, True)}

## recordings-animals

> Whole-head activity of 196 sex-shared and 6 male-specific neurons was studied in 22
> hermaphrodites and 25 males.

In [ ]:
print("Recordings in the dandiset, by sex (subject description):", info.sex.value_counts().to_dict())
print("Recordings with traces of their own, by sex:", Counter(r["sex"] for r, _ in recs))
neur = set(df.neuron)
print("Distinct sex-shared head neurons with traces: %d (paper: 196); male-specific: %d (paper: 6) %s"
      % (len([n for n in neur if chemo.subclass(n) not in chemo.MALE_SPECIFIC]),
         len([n for n in neur if chemo.subclass(n) in chemo.MALE_SPECIFIC]),
         sorted(n for n in neur if chemo.subclass(n) in chemo.MALE_SPECIFIC)))

**Result: reproduced.** The dandiset holds 22 hermaphrodite and 25 male recordings, the paper's
counts, each with traces of its own once the shared table of 20220403-h3 and 20220403-m3 is split
(data check). The traces cover 189 distinct sex-shared neuron names, against the paper's 196, and
the 6 male-specific neurons, as the paper says.

## fig5b-responsive-classes

> Of the 89 head neuron classes, 84 (94%) responded to at least one stimulus in at least one sex.

RESPONSIVE per subclass (pooled responses to a cue against the control, two-sample t-test, per sex,
onset and offset), a canonical class responsive when any of its subclasses is (code Fig5.ipynb).

In [ ]:
rows = []
for (rd, strict), c in C.items():
    can = c["can"]
    rows.append({"exclusion": rd, "threshold": "p < 0.05" if strict else "p <= 0.05", "classes": len(can),
                 "responsive": int(can.responsive.sum()), "dimorphic": int(can.dimorphic.sum()),
                 "dimorphic_and_responsive": int(can.both.sum())})
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
for rd in "ab":
    can = C[(rd, False)]["can"]
    print(f"\nExclusion ({rd}), classes not responsive:", sorted(can.index[~can.responsive]))
# The reference: how many class-cue-window-sex tests are significant at all, and how many would be by chance
rt = X["a"]["rt"]; rt = rt[~rt.subclass.isin(chemo.MALE_SPECIFIC)]
print(f"\nTests (subclass x sex x cue x window): {rt.p.notna().sum()}, significant at p <= 0.05: {(rt.p <= 0.05).mean():.1%} "
      f"(5% expected under the null); per subclass {8 * 2 * 2} tests, so a class with no effect is called "
      f"responsive with probability up to 1 - 0.95^32 = {1 - 0.95 ** 32:.0%} if the tests were independent.")

**Result: reproduced.** 86 of 89 canonical classes are responsive under exclusion (a) and 85 under
(b) (84 expected, tolerance 8.4); the same at p < 0.05. The test is liberal: each subclass has 32
tests, and 22.1% of all tests are significant against 5% expected under the null, so the count is
near its ceiling. This is the paper's test as written.

## fig5b-dimorphic-classes

> Of the 89 head neuron classes, 80 (90%) responded dimorphically to at least one stimulus.

In [ ]:
print(counts.to_string(index=False))
for rd in "ab":
    can = C[(rd, False)]["can"]
    print(f"\nExclusion ({rd}), classes not dimorphic:", sorted(can.index[~can.dimorphic]),
          "| dimorphic but not responsive:", sorted(can.index[can.dimorphic & ~can.responsive]))
dt = X["a"]["dt1"]; dt = dt[~dt.subclass.isin(chemo.MALE_SPECIFIC)]
print(f"\nSex tests (subclass x cue x window): {dt.p.notna().sum()}, significant at p <= 0.05: {(dt.p <= 0.05).mean():.1%}; "
      f"after Benjamini-Hochberg over all of them: {multipletests(dt.p.dropna(), method='fdr_bh')[0].mean():.1%}")

**Result: reproduced.** Dimorphic and responsive (reading a, the code's): 83 classes under
exclusion (a) and 82 under (b); dimorphic alone (reading b): 86 and 85. The claim's 80 has a
tolerance of 8, and all four are within it. 20.9% of the sex tests are significant, 7.6% after
Benjamini-Hochberg over all of them.

## fig5b-layer-counts

> Responsive and dimorphic classes per layer are 23 and 22 of 24 sensory, 34 and 33 of 36 inter,
> 14 and 13 of 15 motor, and 13 and 12 of 14 pharyngeal classes.

The layer of each class is an assumption (see the data check and `chemo.LAYER_CLASSES`).

In [ ]:
expected = {("sensory", "responsive"): (23, 2.3), ("sensory", "dimorphic"): (22, 2.2),
            ("inter", "responsive"): (34, 3.4), ("inter", "dimorphic"): (33, 3.3),
            ("motor", "responsive"): (14, 1.4), ("motor", "dimorphic"): (13, 1.3),
            ("pharyngeal", "responsive"): (13, 1.3), ("pharyngeal", "dimorphic"): (12, 1.2)}
layer_rows = []
for rd in "ab":
    can = C[(rd, False)]["can"]
    for dim_reading, dcol in [("a: dimorphic and responsive", "both"), ("b: dimorphic alone", "dimorphic")]:
        within = 0
        for (layer, kind), (val, tol) in expected.items():
            got = int(can[can.layer == layer]["responsive" if kind == "responsive" else dcol].sum())
            ok = abs(got - val) <= tol
            within += ok
            layer_rows.append({"exclusion": rd, "dimorphism reading": dim_reading, "layer": layer, "kind": kind,
                               "expected": val, "measured": got, "within": ok})
        print(f"exclusion ({rd}), {dim_reading}: {within} of 8 counts within tolerance")
lt = pd.DataFrame(layer_rows)
print(lt.pivot_table(index=["layer", "kind", "expected"], columns=["exclusion", "dimorphism reading"], values="measured").to_string())

**Result: partially reproduced, resting on an assumption.** Under each exclusion and each
dimorphism reading seven of the eight counts are within tolerance; the motor dimorphic count is 15
of 15 against 13 (tolerance 1.3). The layer assignment is assumed (the paper's is not public);
moving a class between layers would change these counts.

## fig3-each-stimulus-mixed

> For every stimulus at least one neuron class responds dimorphically and at least one
> monomorphically.

Per cue, a canonical class is dimorphic when DIMORPHISM gives p <= 0.05 for any of its
subclasses at onset or offset, and monomorphic when it is responsive to that cue (RESPONSIVE) and
no subclass is dimorphic to it.

In [ ]:
def mono_dim(rd):
    c = C[(rd, False)]
    dim = c["can_dim_cue"].reindex(columns=chemo.CUES).fillna(False).astype(bool)
    resp = c["can_resp_cue"].reindex(index=dim.index, columns=chemo.CUES).fillna(False).astype(bool)
    return dim, resp & ~dim

for rd in "ab":
    dim, mono = mono_dim(rd)
    tab = pd.DataFrame({"dimorphic classes": dim.sum(), "monomorphic classes": mono.sum()})
    tab.index = [chemo.CUE_NAMES[i] for i in tab.index]
    print(f"Exclusion ({rd}):")
    print(tab.T.to_string())
    print("cues with at least one of each:", int(((tab["dimorphic classes"] > 0) & (tab["monomorphic classes"] > 0)).sum()), "of 8\n")

**Result: reproduced.** Under both exclusion readings each of the 8 cues has dimorphic classes (22
to 54) and monomorphic classes (14 to 34).

## sensory-dimorphic-and-monomorphic

> Nearly every sensory neuron class responds dimorphically to at least one stimulus and
> monomorphically to at least one other.

In [ ]:
for rd in "ab":
    dim, mono = mono_dim(rd)
    sens = [c for c in chemo.LAYER_CLASSES["sensory"]]
    both = (dim.loc[sens].any(axis=1) & mono.loc[sens].any(axis=1))
    print(f"Exclusion ({rd}): sensory classes dimorphic to one cue and monomorphic to another: {int(both.sum())} of {len(sens)}"
          f"; not: {sorted(both.index[~both])}")
    print("   cues per sensory class, dimorphic / monomorphic:",
          {c: f"{int(dim.loc[c].sum())}/{int(mono.loc[c].sum())}" for c in sens})

**Result: not reproduced, resting on an assumption.** 21 of 24 sensory classes are dimorphic to
one cue and monomorphic to another under exclusion (a) (ASJ, FLP and OLL have no monomorphic cue),
and 20 of 24 under (b), against at least 22. Each sensory class is dimorphic to at least one cue
except IL2 under (b). The sensory classes are those of the assumed layer assignment.

## layers-sensory-most-dimorphic

> Dimorphic and monomorphic responses are less present in interneurons than in sensory neurons,
> and less still in motor neurons.

In [ ]:
for rd in "ab":
    c = C[(rd, False)]
    dim, mono = mono_dim(rd)
    resp = c["can_resp_cue"].reindex(index=dim.index, columns=chemo.CUES).fillna(False).astype(bool)
    lay = dim.index.map(chemo.LAYER)
    ra = dim.groupby(lay).mean().mean(axis=1)
    rb = resp.groupby(lay).mean().mean(axis=1)
    d1 = X[rd]["dt1"]; d1 = d1[~d1.subclass.isin(chemo.MALE_SPECIFIC)]
    rc = (d1.p <= 0.05).groupby(d1.subclass.map(chemo.canonical).map(chemo.LAYER)).mean()
    tab = pd.DataFrame({"(a) class-cue dimorphic": ra, "(b) class-cue responsive": rb, "(c) subclass-cue-window dimorphic": rc})
    print(f"Exclusion ({rd}):")
    print(tab.round(3).to_string())
    for col in tab:
        print(f"   {col}: sensory > inter > motor: {tab.loc['sensory', col] > tab.loc['inter', col] > tab.loc['motor', col]}")

**Result: reproduced.** Under reading (c), the share of subclass-cue-window cells that are
dimorphic is 0.272 in sensory, 0.198 in inter and 0.171 in motor subclasses under exclusion (a)
(0.265, 0.200, 0.144 under (b)). Reading (a) does not order them (motor 0.417 above inter 0.372),
and reading (b) does under exclusion (b) only. The claim holds under one reading, as its criterion
asks; the layers are those of the assumed assignment.

## Single classes: the tests behind the next claims

DIMORPHISM (representation 1) and RESPONSIVE for the subclasses the next claims name, both
exclusion readings. `resp_h` and `resp_m` are the RESPONSIVE p-values in hermaphrodites and
males; `norm_diff` is Figure 3's |male - herm| / (|herm| + |male| + 0.1).

In [ ]:
def cell(rd, sub, cue, window):
    d = X[rd]["dt1"]; r = X[rd]["rt"]
    row = d[(d.subclass == sub) & (d.cue == cue) & (d.window == window)].iloc[0]
    rr = r[(r.subclass == sub) & (r.cue == cue) & (r.window == window)].set_index("sex").p
    return {"exclusion": rd, "class": sub, "cue": chemo.CUE_NAMES[cue], "window": window,
            "n_herm": row.n_herm, "n_male": row.n_male, "mean_herm": row.mean_herm, "mean_male": row.mean_male,
            "norm_diff": row.norm_diff, "p_sex": row.p,
            "resp_h": rr.get("hermaphrodite", np.nan), "resp_m": rr.get("male", np.nan)}


def cells(subs, cues, windows=("onset", "offset")):
    out = pd.DataFrame([cell(rd, s, c, w) for rd in "ab" for s in subs for c in cues for w in windows])
    out["dimorphic"] = out.p_sex <= 0.05
    out["responsive"] = (out.resp_h <= 0.05) | (out.resp_m <= 0.05)
    return out


fmt = {"mean_herm": "{:.3f}".format, "mean_male": "{:.3f}".format, "norm_diff": "{:.2f}".format,
       "p_sex": "{:.2g}".format, "resp_h": "{:.2g}".format, "resp_m": "{:.2g}".format}

## fig3a-osmotic-onset-examples

> At the onset of osmotic stress ASER and ASH respond monomorphically while AWA and AWB respond
> dimorphically.

In [ ]:
t = cells(["AWA", "AWB", "ASER", "ASH"], ["800mM Sorbitol"], ["onset"])
print(t.to_string(index=False, formatters=fmt))
for rd in "ab":
    s = t[t.exclusion == rd].set_index("class")
    ok = (s.loc[["AWA", "AWB"], "dimorphic"].all() and not s.loc[["ASER", "ASH"], "dimorphic"].any()
          and s.loc[["ASER", "ASH"], "responsive"].all())
    print(f"exclusion ({rd}): criterion met: {ok}")

**Result: reproduced.** Under both exclusion readings, at sorbitol onset AWA (p = 4.2e-11,
normalized difference 0.66) and AWB (p = 7.1e-07) are dimorphic, and ASER (p = 0.45) and ASH (p =
0.80) are not, while both respond to sorbitol in each sex (RESPONSIVE p 9.2e-09 or below).

## fig2b-bag-high-odor-monomorphic

> AWA's response to osmotic stress is dimorphic and BAG's response to high odor is monomorphic.

In [ ]:
t = pd.concat([cells(["AWA"], ["800mM Sorbitol"]), cells(["BAG"], ["e-2 IAA"])])
print(t.to_string(index=False, formatters=fmt))
for rd in "ab":
    s = t[t.exclusion == rd].set_index(["class", "window"])
    res = {}
    for w in ("onset", "offset"):
        res[f"({'a' if w == 'onset' else 'b'}) {w}"] = bool(s.loc[("AWA", w), "dimorphic"] and not s.loc[("BAG", w), "dimorphic"]
                                                          and s.loc[("BAG", w), "responsive"])
    res["(c) either for AWA, both for BAG"] = bool(s.loc["AWA"].dimorphic.any() and not s.loc["BAG"].dimorphic.any()
                                                    and s.loc["BAG"].responsive.any())
    print(f"exclusion ({rd}):", res)

**Result: reproduced.** Under every reading and both exclusion readings: AWA's sorbitol response
is dimorphic at onset (p = 4.2e-11) and offset (p = 0.00027), BAG's response to high odor is not
(onset p = 0.78, offset p = 0.38; normalized differences 0.01 and 0.08) while BAG responds
strongly in both sexes.

## fig3a-ash-stimulus-dependent

> ASH responds monomorphically to osmotic and toxic stress but dimorphically to high salt.

In [ ]:
t = cells(["ASH"], ["450mM NaCl", "800mM Sorbitol", "10mM CuSO4"], ["onset"])
print(t.to_string(index=False, formatters=fmt))
for rd in "ab":
    s = t[t.exclusion == rd].set_index("cue")
    ok = (s.loc["high salt", "dimorphic"] and not s.loc[["osmotic", "toxic"], "dimorphic"].any()
          and s.loc[["osmotic", "toxic"], "responsive"].all())
    print(f"exclusion ({rd}): criterion met: {ok}")

**Result: reproduced.** At onset, ASH is dimorphic to high salt (hermaphrodites 1.66, males 0.27,
p = 8e-13) and not to sorbitol (p = 0.80) or CuSO4 (p = 0.13), to both of which it responds in
each sex; the same under exclusion (b).

## fig3a-aser-salt-concentration

> ASER responds dimorphically to low salt but monomorphically to high salt.

In [ ]:
t = cells(["ASER"], ["100mM NaCl", "450mM NaCl"], ["onset"])
print(t.to_string(index=False, formatters=fmt))
for rd in "ab":
    s = t[t.exclusion == rd].set_index("cue")
    ok = s.loc["low salt", "dimorphic"] and not s.loc["high salt", "dimorphic"] and s.loc["high salt", "responsive"]
    print(f"exclusion ({rd}): criterion met: {ok}")

**Result: not reproduced.** At onset ASER's low-salt response falls short of the criterion
(hermaphrodites -0.029, males -0.102, normalized difference 0.31, p = 0.055 under exclusion (a)
and 0.059 under (b)), and its high-salt response is not responsive in either sex (control
comparison p = 0.72 and 0.33), so the monomorphic half fails too (high salt p = 0.89 between
sexes).

## fig2e-i2-dimorphic

> Pharyngeal interneuron I2 responds dimorphically to pheromone, osmotic stress and salt.

In [ ]:
t = cells(["I2"], ["1uM ascr#3", "800mM Sorbitol", "100mM NaCl", "450mM NaCl"])
print(t.to_string(index=False, formatters=fmt))
for rd in "ab":
    s = t[t.exclusion == rd]
    d = s.groupby("cue").dimorphic.any()
    ok = d["pheromone"] and d["osmotic"] and (d["low salt"] or d["high salt"])
    print(f"exclusion ({rd}): dimorphic at onset or offset: {d.to_dict()}; criterion met: {ok}")
    for w in ("onset", "offset"):
        print(f"   {w} only: {s[s.window == w].set_index('cue').dimorphic.to_dict()}")

**Result: reproduced.** I2 is dimorphic to pheromone (onset p = 4.4e-10, offset p = 0.00014),
sorbitol (onset p = 0.034; 0.022 under exclusion (b)) and high salt (onset p = 0.015; 0.010), and
not to low salt (p 0.47 or more). Read at onset only all three hold; at offset only, pheromone
alone.

## pharynx-dimorphic-food-salt-pheromone

> The pharyngeal (enteric) neurons respond dimorphically to food, salt and pheromone.

In [ ]:
phar = [s for s in shared if chemo.LAYER[chemo.canonical(s)] == "pharyngeal"]
for rd in "ab":
    d = X[rd]["dt1"]; d = d[d.subclass.isin(phar) & (d.p <= 0.05)]
    by_cue = d.groupby("cue").subclass.apply(lambda x: sorted(set(x)))
    print(f"exclusion ({rd}): pharyngeal subclasses dimorphic (p <= 0.05, onset or offset) per cue:")
    for cue in ["OP50", "1uM ascr#3", "100mM NaCl", "450mM NaCl"]:
        print(f"   {chemo.CUE_NAMES[cue]}: {by_cue.get(cue, [])}")
    ok = len(by_cue.get("OP50", [])) > 0 and len(by_cue.get("1uM ascr#3", [])) > 0 and \
        (len(by_cue.get("100mM NaCl", [])) > 0 or len(by_cue.get("450mM NaCl", [])) > 0)
    print(f"   criterion met: {ok}")

**Result: reproduced.** Under both exclusion readings pharyngeal subclasses are dimorphic to food
(I4, M2, M5), to pheromone (eight subclasses) and to both salts (four to six subclasses).

## fig2-awa-awb-nociceptive

> Olfactory neurons AWA and AWB also respond to osmotic and toxic stress.

In [ ]:
for rd in "ab":
    r = X[rd]["rt"]
    r = r[r.subclass.isin(["AWA", "AWB"]) & r.cue.isin(["800mM Sorbitol", "10mM CuSO4"])]
    print(f"exclusion ({rd}):")
    print(r.assign(cue=r.cue.map(chemo.CUE_NAMES)).round(4).to_string(index=False))
    calls = (r.p <= 0.05).groupby([r.subclass, r.cue]).any()
    print("   responsive (any sex, any window):", calls.to_dict(), "| criterion met:", bool(calls.all()))

**Result: partially reproduced.** AWB responds to sorbitol and CuSO4 in both sexes (p 0.0046 or
below), and AWA responds to sorbitol in hermaphrodites (onset p = 3.5e-11). AWA does not respond
to CuSO4 in either sex or window (smallest p 0.077 under exclusion (a), 0.085 under (b)). Three of
the four class-cue pairs hold.

## fig2-adf-m2-switch

> ADF responds to food in hermaphrodites but to pheromone in males, and pharyngeal neuron M2 shows
> a similar switch between food and pheromone.

In [ ]:
for rd in "ab":
    r = X[rd]["rt"]; d = X[rd]["dt1"]
    print(f"exclusion ({rd}):")
    res = {}
    for sub in ("ADF", "M2"):
        rr = r[(r.subclass == sub) & r.cue.isin(["OP50", "1uM ascr#3"])]
        resp = (rr.p <= 0.05).groupby([rr.sex, rr.cue]).any().unstack()
        print(f"  {sub} RESPONSIVE (onset or offset):")
        print(rr.assign(cue=rr.cue.map(chemo.CUE_NAMES)).round(4).to_string(index=False))
        h_food, h_pher = resp.loc["hermaphrodite", "OP50"], resp.loc["hermaphrodite", "1uM ascr#3"]
        m_food, m_pher = resp.loc["male", "OP50"], resp.loc["male", "1uM ascr#3"]
        if sub == "ADF":
            ra = h_food and not h_pher and m_pher and not m_food
        else:
            ra = (h_food != h_pher) and (m_food != m_pher) and (h_food != m_food)
        dd = d[(d.subclass == sub) & d.cue.isin(["OP50", "1uM ascr#3"])]
        print(f"  {sub} DIMORPHISM:")
        print(dd.assign(cue=dd.cue.map(chemo.CUE_NAMES)).round(4).to_string(index=False))
        sig = (dd.p <= 0.05).groupby(dd.cue).any()
        # direction: the window with the smaller p for each cue
        best = dd.loc[dd.groupby("cue").p.idxmin()].set_index("cue")
        larger = {c: ("male" if abs(best.loc[c, "mean_male"]) > abs(best.loc[c, "mean_herm"]) else "hermaphrodite") for c in best.index}
        if sub == "ADF":
            rb = sig.all() and larger["OP50"] == "hermaphrodite" and larger["1uM ascr#3"] == "male"
        else:
            rb = sig.all() and larger["OP50"] != larger["1uM ascr#3"]
        print(f"  {sub}: reading (a) {bool(ra)}; reading (b) {bool(rb)} (larger |mean| in the window with the smaller p: {larger})")
        res[sub] = bool(ra or rb)
    print(f"  meets a reading: {res}")

**Result: reproduced.** ADF meets reading (b) under both exclusion readings: its food response is
larger in hermaphrodites (onset 3.93 against 0.76) and its pheromone response larger in males
(5.96 against 0.07), both dimorphic (p below 1e-4). Reading (a) fails for ADF because males also
respond to food (onset p below 1e-4 against control). M2 meets reading (b) under both exclusion
readings (food onset 0.31 in hermaphrodites against 0.01, pheromone onset 0.21 in males against
-0.01, both p below 1e-4) and reading (a) under exclusion (b).

## male-specific-unresponsive

> The male-specific head neuron classes CEM and MCM were predominantly unresponsive to the stimuli.

In [ ]:
for rd in "ab":
    r = X[rd]["rt"]
    r = r[r.subclass.isin(chemo.MALE_SPECIFIC) & (r.sex == "male")]
    tab = r.groupby("subclass").agg(tests=("p", "count"), significant=("p", lambda p: int((p <= 0.05).sum())),
                                    animals=("n_cue", "size"))
    tab["animals"] = [df[(df.subclass == s) & (df.sex == "male")].subject.nunique() for s in tab.index]
    tab["fewer than half"] = tab.significant < tab.tests / 2
    print(f"exclusion ({rd}):")
    print(tab.to_string())
    sigs = r[r.p <= 0.05]
    print("   significant tests:", [(a, chemo.CUE_NAMES[b], c, round(p, 4)) for a, b, c, p in sigs[["subclass", "cue", "window", "p"]].values])
# For reference, the share of significant male tests over the sex-shared sensory and interneuron subclasses
r = X["a"]["rt"]; r = r[(r.sex == "male") & ~r.subclass.isin(chemo.MALE_SPECIFIC)]
print("\nReference, exclusion (a): share of male RESPONSIVE tests with p <= 0.05 over the sex-shared subclasses: "
      f"{(r.p <= 0.05).mean():.1%}")

**Result: reproduced.** Of 16 tests in males, CEMD has 0 significant, CEMV 2 (3 under exclusion
(b)) and MCM 1, all fewer than half; over the sex-shared subclasses 22.3% of male tests are
significant.

## tail-unresponsive

Blocked: the dandiset has no traces of tail neurons (data check). The NeuroPAL segmentations
identify tail neurons, but the trace tables hold head neurons only.

## figs2f-left-right-similar

> Neurons within the same class usually have equivalent activity, with left and right neurons more
> correlated with each other than with neurons of other classes.

Within each animal, the Pearson r between the left and right neuron of each bilateral subclass,
against the mean r of each of them with the neurons of other classes, on (a) the band-passed traces
of CORRELATION and (b) the raw traces (frames 98 on, pairwise complete).

In [ ]:
CORR = {}
for filt in (True, False):
    CORR[filt] = [(r["subject"], r["sex"], chemo.correlation_frame(r, filt, rows)) for r, rows in recs]

lr_rows = []
for filt, cl in CORR.items():
    for subj, sex, c in cl:
        names = [n for n in c.index if n not in chemo.CUES]
        sc = {n: chemo.subclass(n) for n in names}
        for n in names:
            if not n.endswith("L") or sc[n] == n:
                continue
            partner = n[:-1] + "R"
            if partner not in c.index:
                continue
            others = [m for m in names if sc[m] != sc[n]]
            lr = c.loc[n, partner]
            other = np.nanmean(np.r_[c.loc[n, others].values, c.loc[partner, others].values])
            lr_rows.append({"traces": "band-passed" if filt else "raw", "subject": subj, "sex": sex,
                            "subclass": sc[n], "lr": lr, "other": other})
lrt = pd.DataFrame(lr_rows)
summ = lrt.groupby(["traces", "sex", "subclass"])[["lr", "other"]].mean()
summ["lr_higher"] = summ.lr > summ.other
out = summ.groupby(level=[0, 1]).agg(classes=("lr_higher", "size"), lr_higher=("lr_higher", "sum"),
                                       mean_lr=("lr", "mean"), mean_other=("other", "mean"))
out["fraction"] = out.lr_higher / out.classes
print(out.round(3).to_string())
print("\nPer animal: fraction of bilateral subclasses with left-right r above other-class r (band-passed):")
pa = lrt[lrt.traces == "band-passed"].assign(h=lambda x: x.lr > x.other).groupby(["sex", "subject"]).h.mean()
print(pa.groupby(level=0).describe().round(2).to_string())
print("Lowest subclasses (band-passed, mean over sexes, lr - other):",
      (summ.loc["band-passed"].lr - summ.loc["band-passed"].other).groupby(level=1).mean().sort_values().head(8).round(3).to_dict())

**Result: reproduced.** With band-passed traces the mean left-right r exceeds the mean r with
other classes for 77 of 79 bilateral subclasses in hermaphrodites and 80 of 82 in males (mean r
0.44 against 0.04 and 0.05); with raw traces for 72 of 79 and 73 of 82 (0.68 against 0.44, 0.74
against 0.54). MC is the clear exception (left-right r 0.15 below other-class r).

## figs3e-pattern-robust

> Restricting the test to the first cycle or to per-animal averages makes specific neuron-stimulus
> pairs less clearly dimorphic but leaves the pattern of dimorphic neuron classes largely unchanged.

In [ ]:
for rd in "ab":
    sets, npairs = {}, {}
    for rep in (1, 2, 3):
        d = X[rd][f"dt{rep}"]; d = d[~d.subclass.isin(chemo.MALE_SPECIFIC)]
        sets[rep] = set(d[d.p <= 0.05].subclass)
        npairs[rep] = int((d.p <= 0.05).sum())
    base = sets[1]
    all3 = base & sets[2] & sets[3]
    print(f"exclusion ({rd}): dimorphic subclasses rep1 {len(base)}, rep2 {len(sets[2])}, rep3 {len(sets[3])}; "
          f"of rep1's, under rep2 {len(base & sets[2])}, rep3 {len(base & sets[3])}, all three {len(all3)}")
    print(f"   (a) all three: {len(all3) / len(base):.2f}; (b) rep2 {len(base & sets[2]) / len(base):.2f}, rep3 {len(base & sets[3]) / len(base):.2f}")
    print(f"   dimorphic subclass-cue-window pairs: rep1 {npairs[1]}, rep2 {npairs[2]}, rep3 {npairs[3]} (of {len(d)})")
    print(f"   rep1 classes not dimorphic under rep3: {sorted(base - sets[3])}")

**Result: reproduced.** Of the 103 subclasses dimorphic under representation 1 (exclusion (a)), 87
(84%) stay dimorphic under representation 2, 94 (91%) under representation 3, and 80 (78%) under
all three; under exclusion (b) 85%, 96% and 83%. Dimorphic subclass-cue-window pairs fall from 368
to 262 and 254 (exclusion (b): 355 to 256 and 255).

## figs2e-no-order-effect

> Response amplitude does not depend on the position of the stimulus in the presentation order.

Onset responses (avg_during) of cycle 1 against position 1 to 10: (a) linear regression over all
neuron responses; (b) the same on per-animal means over neurons; (c) one-way ANOVA over positions.
Shown for both exclusion readings, all stimuli combined and each stimulus.

In [ ]:
def order_tests(d):
    d = d[(d.cycle == 1) & np.isfinite(d.avg_during)]
    a = stats.linregress(d.position, d.avg_during)
    pa_ = d.groupby(["subject", "position"]).avg_during.mean().reset_index()
    b = stats.linregress(pa_.position, pa_.avg_during)
    c = stats.f_oneway(*[g.avg_during.values for _, g in d.groupby("position")])
    return {"n": len(d), "slope_a": a.slope, "p_a": a.pvalue, "slope_b": b.slope, "p_b": b.pvalue, "F_c": c.statistic, "p_c": c.pvalue}

for rd in "ab":
    e = X[rd]["e"]
    rows_ = [{"stimulus": "all", **order_tests(e)}]
    for s in chemo.STIMULI:
        rows_.append({"stimulus": chemo.CUE_NAMES[s], **order_tests(e[e.stimulus == s])})
    print(f"exclusion ({rd}):")
    print(pd.DataFrame(rows_).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    print(f"   responses without a value in the delivery window (all frames missing), left out: {int((~np.isfinite(e[e.cycle == 1].avg_during)).sum())}")
    e1 = e[e.cycle == 1]
    print("   mean onset response by position (all stimuli):", e1.groupby("position").avg_during.mean().round(4).to_dict())

**Result: blocked (methods underspecified).** For all stimuli combined, under exclusion (b) no
reading finds a dependence on position (regression p = 0.119 on all responses, p = 0.431 on
per-animal means, ANOVA p = 0.056). Under exclusion (a) the regressions find none (p = 0.063 and
0.335) but the ANOVA does (p = 0.034), so under (a) the readings disagree; the mean response by
position ranges from 0.036 to 0.063 with no trend. The claims file blocks the claim when the
exclusion readings give different outcomes (opening comment, EXCLUSION and READINGS). Per
stimulus, the ANOVA is below 0.05 for food, osmotic, low odor, low salt and buffer under both
exclusions, and no slope is below 0.05 except high salt under exclusion (a) (p = 0.011).

## Functional connectivity: the per-sex correlations behind the next claims

CORRELATION (claims file): band-passed traces, Pearson r per pair per animal, and the 8 cue series.
Per subclass pair and sex, reading (b), the Methods': the mean over the left-right combinations
per animal, then the mean over animals, and a t-test on the per-animal values; reading (a), the
code's: all combinations of all animals pooled, values beyond 3 IQR of the quartiles removed, then
the mean and a t-test on the pooled values.

In [ ]:
ents = chemo.CUES + shared                 # 8 cues and 110 sex-shared subclasses
idx = {e: i for i, e in enumerate(ents)}
nE = len(ents)
sex_of = np.array([sex for _, sex, _ in CORR[True]])


def class_pair_values(corr_list):
    """Per animal, the mean r over the left-right combinations of each pair of cues and subclasses,
    and per sex the pooled values of all combinations of all animals."""
    pa = np.full((len(corr_list), nE, nE), np.nan)
    pool = {}
    for a, (subj, sex, c) in enumerate(corr_list):
        names = [n for n in c.index if (n in idx) or (chemo.subclass(n) in idx)]
        lab = np.array([idx[n] if n in idx else idx[chemo.subclass(n)] for n in names])
        v = c.loc[names, names].values
        for i in np.unique(lab):
            mi = lab == i
            for j in np.unique(lab):
                if j <= i:
                    continue
                vals = v[np.ix_(mi, lab == j)].ravel()
                vals = vals[np.isfinite(vals)]
                if len(vals):
                    pa[a, i, j] = vals.mean()
                    pool.setdefault((i, j, sex), []).extend(vals)
    return pa, pool


per_animal_r, pooled = class_pair_values(CORR[True])


def drop3iqr(v):
    v = np.asarray(v)
    q1, q3 = np.percentile(v, [25, 75])
    return v[(v >= q1 - 3 * (q3 - q1)) & (v <= q3 + 3 * (q3 - q1))]


pair_rows = []
H_idx, M_idx = sex_of == "hermaphrodite", sex_of == "male"
for i in range(nE):
    for j in range(i + 1, nE):
        h, m = per_animal_r[H_idx, i, j], per_animal_r[M_idx, i, j]
        h, m = h[np.isfinite(h)], m[np.isfinite(m)]
        ph, pm = pooled.get((i, j, "hermaphrodite"), []), pooled.get((i, j, "male"), [])
        row = {"e1": ents[i], "e2": ents[j], "n_herm_animals": len(h), "n_male_animals": len(m)}
        if len(h) and len(m):
            row.update(herm_b=h.mean(), male_b=m.mean(), p_b=chemo.ttest(h, m))
        if len(ph) and len(pm):
            ph, pm = drop3iqr(ph), drop3iqr(pm)
            row.update(herm_a=ph.mean(), male_a=pm.mean(), p_a=chemo.ttest(ph, pm))
        pair_rows.append(row)
pairs = pd.DataFrame(pair_rows)
pairs["stim_pair"] = pairs.e1.isin(chemo.CUES) | pairs.e2.isin(chemo.CUES)
pairs["both_stim"] = pairs.e1.isin(chemo.CUES) & pairs.e2.isin(chemo.CUES)


def layer_of(e):
    return "stimulus" if e in chemo.CUES else chemo.LAYER[chemo.canonical(e)]


pairs["l1"] = pairs.e1.map(layer_of); pairs["l2"] = pairs.e2.map(layer_of)
for rd in "ab":
    pairs[f"fd_{rd}"] = (pairs[f"herm_{rd}"] - pairs[f"male_{rd}"]).abs()
nn = pairs[~pairs.stim_pair]
print(f"Subclass pairs: {len(nn)} (110 x 109 / 2 = {110 * 109 // 2}); with values in both sexes: {nn.herm_b.notna().sum()} (reading b), {nn.herm_a.notna().sum()} (reading a)")
print("Animals per pair (reading b), hermaphrodites: median %d, min %d; males: median %d, min %d"
      % (nn.n_herm_animals.median(), nn.n_herm_animals.min(), nn.n_male_animals.median(), nn.n_male_animals.min()))
print("Mean r ranges: herm %.2f to %.2f, male %.2f to %.2f (reading b)" % (nn.herm_b.min(), nn.herm_b.max(), nn.male_b.min(), nn.male_b.max()))

## fig4c-functional-connectivity-similar

> Pairwise functional connectivity is highly similar between the sexes (R2 = 0.8 for a linear fit of
> male on hermaphrodite mean correlations).

In [ ]:
for rd in "ba":
    d = nn.dropna(subset=[f"herm_{rd}", f"male_{rd}"])
    lr = stats.linregress(d[f"herm_{rd}"], d[f"male_{rd}"])
    print(f"CORRELATION reading ({rd}): {len(d)} pairs, slope {lr.slope:.3f}, intercept {lr.intercept:.4f}, R2 = {lr.rvalue ** 2:.3f}")
# variant: raw traces (no band-pass), reading b
raw_pa, _ = class_pair_values(CORR[False])
iu = np.triu_indices(nE, 1)
keep_ = (iu[0] >= 8) & (iu[1] >= 8)
rx = np.nanmean(raw_pa[sex_of == "hermaphrodite"], 0)[iu][keep_]
ry = np.nanmean(raw_pa[sex_of == "male"], 0)[iu][keep_]
ok_ = np.isfinite(rx) & np.isfinite(ry)
lr = stats.linregress(rx[ok_], ry[ok_])
both = rx[ok_]
print(f"Variant, raw traces (reading b): {len(both)} pairs, R2 = {lr.rvalue ** 2:.3f}")

**Result: reproduced.** Over the 5995 subclass pairs, a line of male on hermaphrodite mean
correlations has R2 = 0.806 under CORRELATION reading (b) and 0.806 under reading (a) (expected
0.8, tolerance 0.08); with raw traces R2 = 0.763.

## fig4-stimuli-correlate-sensory

> In both sexes stimuli are more correlated or anti-correlated with sensory neurons than with
> interneurons or motor neurons.

In [ ]:
sp = pairs[pairs.stim_pair & ~pairs.both_stim].copy()
sp["layer"] = [l2 if l1 == "stimulus" else l1 for l1, l2 in zip(sp.l1, sp.l2)]
print(f"Stimulus-subclass pairs: {len(sp)} (8 x 110 = 880)")
for rd in "ba":
    rows_ = []
    for sex in ("herm", "male"):
        col = f"{sex}_{rd}"
        g = sp.groupby("layer")[col]
        rows_.append(pd.DataFrame({"sex": sex, "mean |r|": g.apply(lambda x: x.abs().mean()),
                                   "rms r": g.apply(lambda x: np.sqrt((x ** 2).mean())), "pairs": g.count()}))
    tab = pd.concat(rows_)
    print(f"\nCORRELATION reading ({rd}):")
    print(tab.round(4).to_string())
    for sex in ("herm", "male"):
        t_ = tab[tab.sex == sex]
        for m_ in ("mean |r|", "rms r"):
            print(f"   {sex}, {m_}: sensory > inter and > motor: {t_.loc['sensory', m_] > t_.loc['inter', m_] and t_.loc['sensory', m_] > t_.loc['motor', m_]}")

**Result: reproduced.** In each sex and under both measures and both CORRELATION readings, stimuli
correlate more with sensory subclasses (mean |r| 0.037 in hermaphrodites, 0.038 in males) than
with interneurons (0.017, 0.017) or motor neurons (0.016, 0.015). All these correlations are
small.

## fig4-pharynx-uncorrelated

> Pharyngeal neurons show some correlations with each other but are largely uncorrelated with the
> rest of the brain.

In [ ]:
d = nn.copy()
d["group"] = np.where((d.l1 == "pharyngeal") & (d.l2 == "pharyngeal"), "phar-phar",
                      np.where((d.l1 == "pharyngeal") | (d.l2 == "pharyngeal"), "phar-other", "other-other"))
for rd in "ba":
    rows_ = []
    for sex in ("herm", "male"):
        g = d.groupby("group")[f"{sex}_{rd}"]
        rows_.append(pd.DataFrame({"sex": sex, "mean |r|": g.apply(lambda x: x.abs().mean()),
                                   "rms r": g.apply(lambda x: np.sqrt((x ** 2).mean())), "pairs": g.count()}))
    tab = pd.concat(rows_)
    print(f"CORRELATION reading ({rd}):")
    print(tab.round(4).to_string())
    for sex in ("herm", "male"):
        t_ = tab[tab.sex == sex]
        for m_ in ("mean |r|", "rms r"):
            print(f"   {sex}, {m_}: phar-other lowest: {t_.loc['phar-other', m_] < t_.loc['phar-phar', m_] and t_.loc['phar-other', m_] < t_.loc['other-other', m_]}")

**Result: reproduced.** In each sex, under both measures and both CORRELATION readings, pharyngeal
to non-pharyngeal pairs have the lowest mean |r| (0.034 in hermaphrodites, 0.038 in males), below
pharyngeal pairs (0.078, 0.091) and non-pharyngeal pairs (0.110, 0.115).

## fig4-functional-dimorphism-small

> Almost all neuron pairs have a functional dimorphism below 0.4, on a scale of 0 to 2.

In [ ]:
for rd in "ba":
    fd = nn[f"fd_{rd}"].dropna()
    print(f"CORRELATION reading ({rd}): {len(fd)} subclass pairs; fraction below 0.4 = {(fd < 0.4).mean():.4f}; "
          f"max {fd.max():.3f}; 99th percentile {fd.quantile(0.99):.3f}; pairs at 0.4 or above: {int((fd >= 0.4).sum())}")
print("Pairs with significant dimorphism (reading b, p < 0.05): %d of %d; p < 0.01: %d"
      % ((nn.p_b < 0.05).sum(), nn.p_b.notna().sum(), (nn.p_b < 0.01).sum()))

**Result: reproduced.** 5994 of 5995 subclass pairs (99.98%) have a functional dimorphism below
0.4 under CORRELATION reading (b), and the same under reading (a); the largest is 0.414 (0.411).

## Synaptic connectivity: the connectomes behind the next claims (outside inputs)

The paper's code reads a connectome table prepared by S.J. Cook that is not public. Its sources
are public and are read here from the journals' supplements, pinned by the articles' DOIs: the
hermaphrodite of White et al. (1986) as annotated by Cook et al. (2019) and the male of Cook et
al. (2019) (doi:10.1038/s41586-019-1352-7), and datasets 7 and 8 of Witvliet et al. (2021)
(doi:10.1038/s41586-021-03778-8, synapse count matrices). Cook et al. give two forms, synapse
counts (SI 3) and EM serial sections (SI 5, "taking into account both the number of synapses and
the sizes of synapses"); the paper does not say which it used, so both are run. Witvliet's are
counts. CONNECTOME (claims file): each dataset divided by its 95% quantile and multiplied by
White's (here the quantile of the nonzero weights among the recorded head neurons, as zeros would
give a quantile of 0); the hermaphrodite weight is the mean of the three, with its SEM; per
subclass pair the directed weights between the two subclasses' neurons in both directions are
summed (8 for two left-right pairs), the SEMs combined in quadrature; dimorphism degree |Sherm -
Smale| / (Sherm + Smale + 5).

In [ ]:
CONN = {unit: chemo.load_connectomes(url) for unit, url in [("counts", chemo.COOK_SI3), ("sections", chemo.COOK_SI5)]}
recorded = sorted({n for r, rows in recs for n in (r["neuron"] if rows is None else r["neuron"][rows])})
for unit, mats in CONN.items():
    print(unit, {k: v.shape for k, v in mats.items()}, {k: float(v.values.sum()) for k, v in mats.items()})


def directed_stack(mats, neurons):
    names = ["white", "witvliet7", "witvliet8", "male"]
    pre = [n for n in neurons if all(n in mats[k].index for k in names[:3])]
    post = [n for n in neurons if all(n in mats[k].columns for k in names[:3])]
    return pre, post, np.stack([mats[k].reindex(index=pre, columns=post).fillna(0.0).values for k in names])

## connectome-link

> The neuron names of the dandiset's traces can be matched to the cells of the four connectomes.

In [ ]:
mats = CONN["sections"]
link_rows = []
for k, m in mats.items():
    cells = set(m.index) | set(m.columns)
    found = [n for n in recorded if n in cells]
    link_rows.append({"connectome": k, "cells": len(cells), "recorded neurons": len(recorded), "matched": len(found),
                      "fraction": len(found) / len(recorded),
                      "unmatched (examples)": ", ".join(sorted(set(recorded) - cells)[:12])})
link = pd.DataFrame(link_rows)
print(link.to_string(index=False))
sub_found = {s: all(any(chemo.subclass(n) == s and n in (set(mats[k].index) | set(mats[k].columns)) for n in recorded)
                    for k in ("white", "witvliet7", "witvliet8", "male")) for s in shared}
print(f"\nSex-shared subclasses with a neuron in all four connectomes: {sum(sub_found.values())} of {len(shared)}")
print("Not in all four, by layer:", Counter(chemo.LAYER[chemo.canonical(s)] for s, v in sub_found.items() if not v))
print("Not in all four:", sorted(s for s, v in sub_found.items() if not v))
vnc = {s for s in shared if chemo.canonical(s) in ("AS", "DA", "DB", "DD", "VA", "VB", "VD")}
outside = [s for s in shared if chemo.LAYER[chemo.canonical(s)] != "pharyngeal" and s not in vnc]
m_out = sum(sub_found[s] for s in outside)
print(f"Outside the pharynx and the ventral cord: {m_out} of {len(outside)} subclasses in all four ({m_out / len(outside):.1%}); "
      f"missing: {sorted(s for s in outside if not sub_found[s])}")

**Result: reproduced.** The connectomes use the dandiset's NeuroPAL names. Outside the pharynx and
the ventral cord, which the paper says are not in the connectome data, 83 of 86 sex-shared
subclasses (96.5%) have neurons in all four connectomes; AVG, SABD and SABV are not in Witvliet's
datasets.

## connectome-normalization-sem

> Normalizing the three hermaphrodite connectomes by their 95% quantiles cut the mean SEM across
> them from 1.14 (SD 2.07) to 0.62 (SD 0.87).

SEM (ddof 1) per directed synapse over the three hermaphrodite datasets, over (a) the code's set,
synapses between recorded head neurons, and (b) every directed pair with a synapse in at least one
hermaphrodite dataset, among the cells all three contain; each with Cook's counts or sections.

In [ ]:
sem_rows = []
for unit, mats in CONN.items():
    allcells = sorted(set.intersection(*[set(mats[k].index) | set(mats[k].columns) for k in ("white", "witvliet7", "witvliet8")]))
    for setname, neurons in [("(a) recorded head neurons", recorded), ("(b) all cells of the three", allcells)]:
        pre, post, S = directed_stack(mats, neurons)
        herm = S[:3]
        mask = (herm > 0).any(0)
        v = herm[:, mask]
        q = [np.quantile(x[x > 0], 0.95) for x in v]
        vn = np.stack([x / qq * q[0] for x, qq in zip(v, q)])
        sem0, sem1 = stats.sem(v, axis=0, ddof=1), stats.sem(vn, axis=0, ddof=1)
        sem_rows.append({"White and male": unit, "set": setname, "synapses": int(mask.sum()),
                         "q95 (White, W7, W8)": np.round(q, 1).tolist(),
                         "mean SEM before": sem0.mean(), "SD before": sem0.std(ddof=1),
                         "mean SEM after": sem1.mean(), "SD after": sem1.std(ddof=1)})
semt = pd.DataFrame(sem_rows)
print(semt.round(3).to_string(index=False))
exp = [1.14, 2.07, 0.62, 0.87]
for _, r in semt.iterrows():
    got = [r["mean SEM before"], r["SD before"], r["mean SEM after"], r["SD after"]]
    print(r["White and male"], r["set"], "within tolerance:", [abs(g - e) <= 0.1 * e for g, e in zip(got, exp)])

**Result: not reproduced.** No reading puts all four values within tolerance (table above). With
Cook's EM sections over every synapse the mean SEM before normalization is 1.143, the paper's
1.14, but its SD is 1.68 against 2.07, and normalization raises the mean SEM to 1.32 where the
paper reports a fall to 0.62. Normalizing to White's quantile scales the Witvliet counts up. The
paper's own table (prepared by S.J. Cook) is not public, and its units may differ.

In [ ]:
def subclass_synapses(unit):
    """Per pair of sex-shared subclasses: summed hermaphrodite (mean of three, normalized) and male
    (normalized) weights over both directions, the combined SEM, and the dimorphism degree."""
    mats = CONN[unit]
    pre, post, S = directed_stack(mats, recorded)
    herm, male = S[:3], S[3]
    q = [np.quantile(x[x > 0], 0.95) for x in S]
    Sn = np.stack([x / qq * q[0] for x, qq in zip(S, q)])
    hm, hsem, mm = Sn[:3].mean(0), stats.sem(Sn[:3], axis=0, ddof=1), Sn[3]
    pre_sub = np.array([chemo.subclass(n) for n in pre]); post_sub = np.array([chemo.subclass(n) for n in post])
    present = sorted(set(pre_sub) & set(post_sub) & set(shared))
    rows_ = []
    for i, a in enumerate(present):
        for b in present[i + 1:]:
            m1 = np.ix_(pre_sub == a, post_sub == b); m2 = np.ix_(pre_sub == b, post_sub == a)
            sh = hm[m1].sum() + hm[m2].sum(); sm = mm[m1].sum() + mm[m2].sum()
            se = np.sqrt((hsem[m1] ** 2).sum() + (hsem[m2] ** 2).sum())
            rows_.append({"s1": a, "s2": b, "S_herm": sh, "S_male": sm, "comb_sem": se,
                          "syn_dim": abs(sh - sm) / (sh + sm + 5), "syn_dim3": abs(sh - sm) / (sh + sm + 3)})
    return pd.DataFrame(rows_)


SYN = {unit: subclass_synapses(unit) for unit in CONN}
fpairs = nn.set_index(["e1", "e2"])


def with_function(syn):
    key = [(a, b) if (a, b) in fpairs.index else (b, a) for a, b in zip(syn.s1, syn.s2)]
    f = fpairs.reindex(key)
    return syn.assign(func_dim=f.fd_b.values, p_func=f.p_b.values, herm_r=f.herm_b.values, male_r=f.male_b.values,
                      func_dim_a=f.fd_a.values)


SYN = {u: with_function(s) for u, s in SYN.items()}
for u, s in SYN.items():
    print(f"{u}: {len(s)} subclass pairs present in the connectomes; with any synapse {int(((s.S_herm > 0) | (s.S_male > 0)).sum())}; "
          f"S_herm max {s.S_herm.max():.1f}, S_male max {s.S_male.max():.1f}")

## fig4d-functional-synaptic-uncorrelated

> Functional and synaptic dimorphisms are uncorrelated.

In [ ]:
for u, s in SYN.items():
    d = s.dropna(subset=["func_dim"])
    for name, col, sub in [("regularization 5", "syn_dim", d), ("variant: regularization 3, pairs with both sums below 3 dropped", "syn_dim3",
                                                                d[(d.S_herm >= 3) | (d.S_male >= 3)])]:
        lr = stats.linregress(sub[col], sub.func_dim)
        print(f"{u}, {name}: {len(sub)} pairs, slope {lr.slope:.4f}, R2 = {lr.rvalue ** 2:.4f}, p = {lr.pvalue:.3g}")
    lr = stats.linregress(d.syn_dim, d.func_dim_a)
    print(f"{u}, functional dimorphism by CORRELATION reading (a): slope {lr.slope:.4f}, R2 = {lr.rvalue ** 2:.4f}, p = {lr.pvalue:.3g}")

**Result: reproduced.** Over the 3403 pairs of the 83 subclasses present in all four connectomes,
the regression of functional on synaptic dimorphism has slope -0.0028, R2 = 0.0001, p = 0.641 with
Cook's synapse counts, and slope -0.0005, p = 0.905 with Cook's EM sections. The code's variant
(regularization 3, pairs with both sums below 3 dropped) gives p = 0.0092 with counts (400 pairs,
slope -0.030, R2 = 0.017) and p = 0.085 with sections; the slope is negative in both, so it finds
no positive relationship either.

## fig4f-synaptic-top-not-functional

> The neuron pairs with the highest synaptic dimorphisms are functionally uncorrelated in both
> sexes and show little to no functional dimorphism.

In [ ]:
SIX = [("RIM", "SAAD"), ("AIB", "AIM"), ("URYD", "RMDV"), ("AVK", "RMF"), ("URX", "RIR"), ("ASH", "AVA")]
for u, s in SYN.items():
    top = s[s.comb_sem < 5].sort_values("syn_dim", ascending=False).head(36)
    six = pd.concat([s[((s.s1 == a) & (s.s2 == b)) | ((s.s1 == b) & (s.s2 == a))] for a, b in SIX])
    print(f"{u}: top 36 by synaptic dimorphism (combined SEM < 5); syn_dim {top.syn_dim.min():.2f} to {top.syn_dim.max():.2f}")
    for name, g in [("top 36", top), ("the six of Figure 4F", six)]:
        print(f"   {name}: {len(g)} pairs; no functional dimorphism (a) p > 0.05: {int((g.p_func > 0.05).sum())}, "
              f"(b) p >= 0.01: {int((g.p_func >= 0.01).sum())}; mean |r| herm {g.herm_r.abs().mean():.3f}, male {g.male_r.abs().mean():.3f}; "
              f"functional dimorphism median {g.func_dim.median():.3f}")
    print(six[["s1", "s2", "S_herm", "S_male", "comb_sem", "syn_dim", "herm_r", "male_r", "func_dim", "p_func"]].round(3).to_string(index=False))

**Result: reproduced.** With counts, 35 of the top 36 synaptically dimorphic pairs have no
functional dimorphism under either reading (p > 0.05, p >= 0.01); with sections 30 and 34. All six
pairs of Figure 4F are among the most dimorphic pairs here (dimorphism 0.75 to 0.88 with counts)
and none has a functional dimorphism (p 0.071 to 0.881). Their mean |r| is 0.044 in hermaphrodites
and 0.090 in males; over the top 36 it is 0.145 and 0.156 (counts), reported descriptively.

## fig4g-functional-top-not-synaptic

> The neuron pairs with the highest functional dimorphisms are mostly not synaptically connected,
> and where connected their connectivity is essentially not dimorphic.

In [ ]:
sens = {s for s in shared if chemo.LAYER[chemo.canonical(s)] == "sensory"}
for u, s in SYN.items():
    in_conn = {(a, b) for a, b in zip(s.s1, s.s2)} | {(b, a) for a, b in zip(s.s1, s.s2)}
    syn_key = {**{(a, b): r for a, b, r in zip(s.s1, s.s2, s.itertuples())}, **{(b, a): r for a, b, r in zip(s.s1, s.s2, s.itertuples())}}
    allp = pairs[pairs.p_b < 0.01].sort_values("fd_b", ascending=False)
    ok_pair = lambda r: (not r.stim_pair) and not (r.e1 in sens and r.e2 in sens) and (r.e1, r.e2) in in_conn
    for order, top in [("(i) exclusions first", allp[[ok_pair(r) for r in allp.itertuples()]].head(36)),
                       ("(ii) top 36 first", allp.head(36))]:
        if order.startswith("(ii)"):
            top = top[[(not r.stim_pair) and (r.e1, r.e2) in in_conn for r in top.itertuples()]]
        recs_ = [syn_key[(r.e1, r.e2)] for r in top.itertuples()]
        sh = np.array([x.S_herm for x in recs_]); sm = np.array([x.S_male for x in recs_])
        for cname, notcon in [("(a) zero in both", (sh == 0) & (sm == 0)), ("(b) both below 3", (sh < 3) & (sm < 3))]:
            con = ~notcon
            notdim = np.abs(sh - sm) < 5
            met = notcon.sum() > len(top) / 2 and (notdim[con].sum() > con.sum() / 2 if con.sum() else True)
            print(f"{u}, {order}, not connected {cname}: {len(top)} pairs, not connected {int(notcon.sum())}, connected {int(con.sum())}, "
                  f"of which not dimorphic (|difference| < 5) {int(notdim[con].sum())}; criterion met: {met}")

**Result: blocked (methods underspecified).** With Cook's counts the criterion is met under both
orders of exclusion: (i) 21 of 36 pairs with zero weight in both sexes (30 with both below 3), and
13 of the 15 connected ones (4 of 6) not dimorphic; (ii) 22 of 27 below 3 in both sexes, 4 of 5
connected not dimorphic. With Cook's EM sections order (i) meets it (20 of 36 unconnected, 13 of
16 connected not dimorphic) and order (ii) does not (10 or 21 of 27 unconnected, 14 of 17 and 3 of
6 connected not dimorphic). The outcome depends on which form of the Cook connectomes the paper
used, which it does not say.

## figs20-multihop-no-match

> Indirect multi-hop synaptic connections are stronger between all pairs but still fail to match
> functional dimorphisms.

In [ ]:
from scipy.sparse.csgraph import dijkstra

for u, s in SYN.items():
    nodes = sorted(set(s.s1) | set(s.s2)); ix = {n: i for i, n in enumerate(nodes)}
    out = {}
    for sex, col in (("herm", "S_herm"), ("male", "S_male")):
        W = np.zeros((len(nodes), len(nodes)))
        for a, b, w in zip(s.s1, s.s2, s[col]):
            if w > 0:
                W[ix[a], ix[b]] = W[ix[b], ix[a]] = 1.0 / w
        D = dijkstra(W, directed=False)
        with np.errstate(divide="ignore"):
            out[sex] = np.where(np.isfinite(D) & (D > 0), 1.0 / D, 0.0)
    ih = np.array([out["herm"][ix[a], ix[b]] for a, b in zip(s.s1, s.s2)])
    im = np.array([out["male"][ix[a], ix[b]] for a, b in zip(s.s1, s.s2)])
    ind = s.assign(I_herm=ih, I_male=im, ind_dim=np.abs(ih - im) / (ih + im + 5)).dropna(subset=["func_dim"])
    stronger = ((ind.I_herm >= ind.S_herm - 1e-9) & (ind.I_male >= ind.S_male - 1e-9)).mean()
    lr = stats.linregress(ind.ind_dim, ind.func_dim)
    print(f"{u}: {len(ind)} pairs; indirect strength at least the direct one in both sexes for {stronger:.1%}; "
          f"median indirect herm {np.median(ih):.2f} against direct {s.S_herm.median():.2f}; "
          f"regression of functional on indirect dimorphism: slope {lr.slope:.4f}, R2 = {lr.rvalue ** 2:.4f}, p = {lr.pvalue:.3g}")

**Result: reproduced.** Indirect (shortest-path) strengths are at least the direct ones for all
3403 pairs in both sexes (median indirect hermaphrodite strength 5.85 with counts where the median
direct one is 0), and the regression of functional on indirect dimorphism has p = 0.215 with
counts and p = 0.963 with sections.

## Motor state from AVA

Methods, "Classifying motor state": the raw AVA trace is corrected for bleaching with an
exponential fit of its baseline; the 5 s before each stimulus is a transition if the activity
changes by more than 20%, otherwise reverse when its mean is above the global mean of the
recording and forward when below. The claims file leaves open (OPEN, all combinations run) which
AVA trace (AVAL, AVAR or their mean), what "changes by more than 20%" means ((max - min) / mean of
the window, or |last - first| / first) and whether the fit is divided or subtracted. Here the
exponential is fitted to the whole trace (a * exp(-t / tau) + c, least squares), the subtracted
variant adds back the fit's mean so that a 20% change keeps a scale, and the 5 s window is the 13
frames before the onset frame (5 s x 2.66 volumes/s, a standard rounding). All 12 combinations are
run under both exclusion readings. The one printed first ("main") is the mean of AVAL and AVAR,
divided by the fit, with (max - min) / mean: dividing is the usual bleaching correction and the
Methods speak of AVA as one trace.

In [ ]:
PRE = 13


def exp_fit(y):
    t = np.arange(len(y), dtype=float)
    ok = np.isfinite(y)
    tt, yy = t[ok], y[ok]
    try:
        p, _ = optimize.curve_fit(lambda t, a, k, c: a * np.exp(-k * t) + c, tt, yy,
                                  p0=(max(yy[:200].mean() - yy[-200:].mean(), 1e-3), 1e-3, yy[-200:].mean()),
                                  bounds=([0, 0, -np.inf], [np.inf, 1, np.inf]), maxfev=5000)
        return p[0] * np.exp(-p[1] * t) + p[2], True
    except Exception:
        b = np.polyfit(tt, yy, 1)
        return np.polyval(b, t), False


def states(rec, rows, which, change, mode):
    names = rec["neuron"] if rows is None else rec["neuron"][rows]
    A = rec["activity"] if rows is None else rec["activity"][rows]
    tr = [A[names.index(n)] for n in ("AVAL", "AVAR") if n in names and which in ("mean", n)]
    if not tr:
        return None, None
    y = np.nanmean(np.vstack(tr), 0) if len(tr) > 1 else tr[0]
    fit, ok = exp_fit(y)
    corr_ = y / fit if mode == "divide" else y - fit + np.nanmean(fit)
    gmean = np.nanmean(corr_)
    out = []
    for j, on in enumerate(chemo.ON_STIM):
        w = corr_[on - PRE:on]
        w = w[np.isfinite(w)]
        if len(w) < 2:
            out.append("unknown"); continue
        ch = (w.max() - w.min()) / abs(w.mean()) if change == "range" else abs(w[-1] - w[0]) / abs(w[0]) if w[0] != 0 else np.inf
        out.append("transition" if ch > 0.2 else ("reverse" if w.mean() > gmean else "forward"))
    return out, ok


READINGS = [(w, c, m) for w in ("mean", "AVAL", "AVAR") for c in ("range", "endpoints") for m in ("divide", "subtract")]
MAIN = ("mean", "range", "divide")
STATE = {}
fit_fail = Counter()
for rdg in READINGS:
    rows_ = []
    for r, rows in recs:
        st, ok = states(r, rows, *rdg)
        if st is None:
            continue
        fit_fail[ok] += 1
        for j, s in enumerate(st):
            rows_.append({"subject": r["subject"], "cycle": j // 10 + 1, "position": j % 10 + 1, "state": s})
    STATE[rdg] = pd.DataFrame(rows_)
print("Exponential fits converged / fell back to a line (over all readings and animals):", dict(fit_fail))
tab = pd.DataFrame({" / ".join(k): v.state.value_counts() for k, v in STATE.items()}).T.fillna(0).astype(int)
print("Presentations per state, by reading (AVA trace / change measure / fit):")
print(tab.to_string())
print("Animals with an AVA trace (mean reading):", STATE[MAIN].subject.nunique())

In [ ]:
def interaction_p(y, sx, st):
    n = len(y)
    X1 = np.column_stack([np.ones(n), sx, st])
    X2 = np.column_stack([X1, sx * st])
    r1 = y - X1 @ np.linalg.lstsq(X1, y, rcond=None)[0]
    r2 = y - X2 @ np.linalg.lstsq(X2, y, rcond=None)[0]
    rss1, rss2 = r1 @ r1, r2 @ r2
    if rss2 <= 0 or n <= 4:
        return np.nan
    F = (rss1 - rss2) / (rss2 / (n - 4))
    return float(stats.f.sf(F, 1, n - 4))


def state_tables(rd, rdg):
    e = X[rd]["e"].merge(STATE[rdg], on=["subject", "cycle", "position"], how="inner")
    e = e[e.state.isin(["forward", "reverse"]) & e.stimulus.isin(chemo.CUES) & ~e.subclass.isin(chemo.MALE_SPECIFIC)]
    rows_ = []
    for (sub, cue), g in e.groupby(["subclass", "stimulus"]):
        for win, col in chemo.WINDOW_COL.items():
            gg = g[np.isfinite(g[col])]
            cnt = gg.groupby(["sex", "state"]).size()
            if len(cnt) < 4 or cnt.min() < 2:
                rows_.append({"subclass": sub, "cue": cue, "window": win, "p_int": np.nan}); continue
            p = interaction_p(gg[col].values, (gg.sex == "male").values.astype(float), (gg.state == "reverse").values.astype(float))
            rows_.append({"subclass": sub, "cue": cue, "window": win, "p_int": p})
    t = pd.DataFrame(rows_)
    t["q_int"] = np.nan
    for win in ("onset", "offset"):
        m = (t.window == win) & t.p_int.notna()
        t.loc[m, "q_int"] = multipletests(t.loc[m, "p_int"], method="fdr_bh")[1]
    t["layer"] = t.subclass.map(lambda s: chemo.LAYER[chemo.canonical(s)])
    return e, t

ST = {(rd, rdg): state_tables(rd, rdg) for rd in "ab" for rdg in READINGS}
print("ANOVA cells tested / untestable (fewer than 2 responses in a sex x state cell), main reading, exclusion (a):",
      int(ST[("a", MAIN)][1].p_int.notna().sum()), "/", int(ST[("a", MAIN)][1].p_int.isna().sum()))

## figs3fg-state-dependence-interneurons

> Motor-state-dependent sexual dimorphism is more prominent in interneurons than in sensory or motor
> neurons.

In [ ]:
rows_ = []
for (rd, rdg), (e, t) in ST.items():
    t = t[t.p_int.notna()]
    for thr_name, col, thr in [("q < 0.05", "q_int", 0.05), ("p < 0.05", "p_int", 0.05)]:
        fr = (t[col] < thr).groupby(t.layer).mean()
        rows_.append({"exclusion": rd, "AVA": rdg[0], "change": rdg[1], "fit": rdg[2], "measure": thr_name,
                      "sensory": fr.get("sensory"), "inter": fr.get("inter"), "motor": fr.get("motor"),
                      "pharyngeal": fr.get("pharyngeal"),
                      "inter highest": bool(fr.get("inter") > fr.get("sensory") and fr.get("inter") > fr.get("motor"))})
sfg = pd.DataFrame(rows_)
print(sfg.round(4).to_string(index=False))
for meas in ("q < 0.05", "p < 0.05"):
    s = sfg[sfg.measure == meas]
    print(f"{meas}: interneurons highest in {int(s['inter highest'].sum())} of {len(s)} reading combinations")
print("Main reading:", sfg[(sfg.AVA == MAIN[0]) & (sfg.change == MAIN[1]) & (sfg.fit == MAIN[2])].round(4).to_string(index=False))

**Result: blocked (methods underspecified).** The interneurons have the highest share of
state-dependent dimorphic cells in 22 of the 24 combinations of the open details, under both
measures; the exception is AVAR alone, divided by the fit, with (max - min) / mean, under both
exclusion readings. Under the reading this notebook would adopt (mean of AVAL and AVAR, fit
divided, (max - min) / mean) the share at q < 0.05 is 0.62% in interneurons against 0% in sensory
and motor neurons (exclusion (a)), and at p < 0.05 11.6% against 8.6% and 9.7%. The claims file
blocks the claim when the motor-state readings disagree (opening comment, READINGS). All q < 0.05
shares are under 2%, so the ordering rests on few cells either way.

## figs2c-aib-state

> The structure of AIB's sexual dimorphism to high salt or pheromone differs between forward and
> backward motor states.

In [ ]:
rows_ = []
for (rd, rdg), (e, t) in ST.items():
    a = e[e.subclass == "AIB"]
    for cue in ("450mM NaCl", "1uM ascr#3"):
        for win, col in chemo.WINDOW_COL.items():
            g = a[(a.stimulus == cue) & np.isfinite(a[col])]
            pf = chemo.ttest(g[(g.state == "forward") & (g.sex == "hermaphrodite")][col], g[(g.state == "forward") & (g.sex == "male")][col])
            pr = chemo.ttest(g[(g.state == "reverse") & (g.sex == "hermaphrodite")][col], g[(g.state == "reverse") & (g.sex == "male")][col])
            pi = t[(t.subclass == "AIB") & (t.cue == cue) & (t.window == win)].p_int.iloc[0]
            met = (pi < 0.05) or ((pf <= 0.05) != (pr <= 0.05) and np.isfinite(pf) and np.isfinite(pr))
            rows_.append({"exclusion": rd, "reading": "/".join(rdg), "cue": chemo.CUE_NAMES[cue], "window": win,
                          "n": len(g), "p_forward": pf, "p_reverse": pr, "p_interaction": pi, "met": met})
aib = pd.DataFrame(rows_)
by = aib.groupby(["exclusion", "reading"]).met.any()
print(aib[aib.reading == "/".join(MAIN)].round(4).to_string(index=False))
print("\nCriterion met (any cue and window) per reading combination:")
print(by.unstack(0).to_string())
print(f"met in {int(by.sum())} of {len(by)} combinations")

**Result: reproduced.** Under every one of the 24 reading combinations
the criterion is met. Under the main reading AIB's high-salt onset response is dimorphic in
reverse (p below 1e-4) but not in forward (p = 0.12), with interaction p = 0.0027, and its
pheromone onset response has interaction p = 0.0012.

## figs8-state-insensitive

> Average responses are mostly insensitive to the animal's motor state, and many dimorphic sensory
> responses persist after conditioning on it.

In [ ]:
FIG2 = ["ADF", "ADL", "ASEL", "ASER", "ASH", "ASI", "AWA", "AWB", "BAG", "AIA", "AIB", "AVE", "AVH", "RIA", "RIR",
        "DB1", "VB2", "M2", "I2", "MC", "M3"]
SENS2 = FIG2[:9]
rows_ = []
for (rd, rdg), (e, t) in ST.items():
    e2 = e[e.subclass.isin(FIG2)]
    ps = []
    for (sub, cue, sex), g in e2.groupby(["subclass", "stimulus", "sex"]):
        for win, col in chemo.WINDOW_COL.items():
            ps.append(chemo.ttest(g[g.state == "forward"][col], g[g.state == "reverse"][col]))
    ps = np.array(ps); ps = ps[np.isfinite(ps)]
    d1 = X[rd]["dt1"]
    dimo = d1[d1.subclass.isin(SENS2) & (d1.p <= 0.05)]
    keep = 0
    for _, row in dimo.iterrows():
        g = e2[(e2.subclass == row.subclass) & (e2.stimulus == row.cue)]
        col = chemo.WINDOW_COL[row.window]
        pf = chemo.ttest(g[(g.state == "forward") & (g.sex == "hermaphrodite")][col], g[(g.state == "forward") & (g.sex == "male")][col])
        pr = chemo.ttest(g[(g.state == "reverse") & (g.sex == "hermaphrodite")][col], g[(g.state == "reverse") & (g.sex == "male")][col])
        keep += (pf <= 0.05) or (pr <= 0.05)
    rows_.append({"exclusion": rd, "reading": "/".join(rdg), "state tests": len(ps), "share p > 0.05": np.mean(ps > 0.05),
                  "sensory dimorphic": len(dimo), "stay dimorphic in a state": keep, "share kept": keep / max(len(dimo), 1)})
s8 = pd.DataFrame(rows_)
s8["met"] = (s8["share p > 0.05"] > 0.5) & (s8["share kept"] > 0.5)
print(s8.round(3).to_string(index=False))
print(f"met in {int(s8.met.sum())} of {len(s8)} combinations")

**Result: reproduced.** Under the main reading 81.7% of forward against
reverse comparisons over the Figure 2 classes have p > 0.05, and 37 of 48 dimorphic sensory
responses (77%) stay dimorphic within the forward or the reverse state; all 24 reading
combinations meet the criterion (79.8% to 87.2%, and 69% to 80%).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| recordings-animals | dandiset | reproduced |
| fig5b-responsive-classes | dandiset | reproduced |
| fig5b-dimorphic-classes | dandiset | reproduced |
| fig5b-layer-counts | dandiset | partially reproduced (layer assignment assumed) |
| fig3-each-stimulus-mixed | dandiset | reproduced |
| sensory-dimorphic-and-monomorphic | dandiset | not reproduced (layer assignment assumed) |
| layers-sensory-most-dimorphic | dandiset | reproduced |
| fig3a-osmotic-onset-examples | dandiset | reproduced |
| fig2b-bag-high-odor-monomorphic | dandiset | reproduced |
| fig3a-ash-stimulus-dependent | dandiset | reproduced |
| fig3a-aser-salt-concentration | dandiset | not reproduced |
| fig2e-i2-dimorphic | dandiset | reproduced |
| pharynx-dimorphic-food-salt-pheromone | dandiset | reproduced |
| fig2-awa-awb-nociceptive | dandiset | partially reproduced |
| fig2-adf-m2-switch | dandiset | reproduced |
| male-specific-unresponsive | dandiset | reproduced |
| tail-unresponsive | dandiset | blocked: data missing |
| figs2f-left-right-similar | dandiset | reproduced |
| figs3e-pattern-robust | dandiset | reproduced |
| figs2e-no-order-effect | dandiset | blocked: methods underspecified |
| figs3fg-state-dependence-interneurons | dandiset | blocked: methods underspecified |
| figs2c-aib-state | dandiset | reproduced |
| figs8-state-insensitive | dandiset | reproduced |
| fig4c-functional-connectivity-similar | dandiset | reproduced |
| fig4-stimuli-correlate-sensory | dandiset | reproduced |
| fig4-pharynx-uncorrelated | dandiset | reproduced |
| fig4-functional-dimorphism-small | dandiset | reproduced |
| connectome-link | with inputs | reproduced |
| connectome-normalization-sem | with inputs | not reproduced |
| fig4d-functional-synaptic-uncorrelated | with inputs | reproduced |
| fig4f-synaptic-top-not-functional | with inputs | reproduced |
| fig4g-functional-top-not-synaptic | with inputs | blocked: methods underspecified |
| figs20-multihop-no-match | with inputs | reproduced |
| figs1-behavior | dandiset | blocked: data missing |

How the analysis differs from the paper's. Responses, exclusion, the class tests and the
correlations follow the claims file and the paper's code. The code's identification-confidence
filter cannot be applied, as the dandiset has no confidence values; its time ranges to ignore are
already blank in the traces, and its ganglia to ignore cannot be checked. The layer of each class
is assumed (24, 36, 15 and 14 classes, the paper's totals), as the code's class file is not
public. The motor state uses an exponential fit to the whole AVA trace and a 13-frame window, with
every open detail of the Methods run. The connectomes are rebuilt from their published sources, as
the paper's merged table is not public, with Cook's synapse counts and EM sections both run; the
male connectome is normalized by the same quantile rule as the hermaphrodite ones.

What was found about the dandiset.

- Files 20220403-h3 and 20220403-m3 hold the same 382-row trace table, h3's traces followed by
  m3's; the blocks are told apart only by the responses and the male-specific labels.
- The subject `sex` field is "O" in every file; the sex is only in the description and the
  subject id. The strain field is "[]" though the paper names OH16230.
- The stimulus annotation times are labelled seconds but are frame numbers, and lie 1 to 3 frames
  after the onset frames the paper's code uses.
- The traces have no unit or description of how they were computed; blocks of frames are blank
  (the first 96 frames in 13 recordings, and other ranges), without a note of why.
- No tail neuron has a trace, though the NeuroPAL segmentations identify tail neurons.
- There is no identification confidence, which the paper's code uses to select neurons.